In [1]:
using DifferentialEquations
using Plots
using SciMLOperators
using Rotations
using StaticArrays
using DoubleFloats
using LinearAlgebra
using GaussQuadrature
using LegendrePolynomials
using BenchmarkTools
using Profile
using ProfileVega

In [2]:
const numtype = Float64
const SV = SVector{3,numtype}
const neutrongyro::numtype = -1.83247172e4
const he3gyro::numtype = -2.037894585e4
const w::numtype = 6283.18530718
const B1::numtype = 4.0289106454828055e-1
const B0::numtype = 0.050

0.05

In [3]:
function flipfill!(f)
    s, k = size(f)
    for i=((s+1)÷2 + 1):s
        for n=1:k
            f[i,n] = (-1)^(n+1) * f[s-i+1,n]
        end
    end
end

function order4CFET()
    f = zeros(numtype,3,4)
    f[1,1] = 11/40
    f[1,2] = 20/87
    f[2,1] = 9/20
    flipfill!(f)
    f
end

function order6CFET()
    # Scheme with 4 exponentials
    f = zeros(numtype,4,4)
    f[1,1] = 1/2 + (5400 - 600 * sqrt(6))^(1/3)/60 + ((9 + sqrt(6))/5)^(1/3)/(2 * 3^(2/3))
    f[1,2] = f[1,1] - (2/3) * f[1,1]^2
    f[1,3] = 1/(10 - 10 * f[1,1])
    f[2,1] = 1/2 - f[1,1]
    f[2,2] = (1 - 4 * f[1,1] + 2 * f[1,1]^2)/3
    f[2,3] = -f[1,3]
    flipfill!(f)
    f
end

function order6CFETB()
    # Scheme with 5 exponentials
    f = zeros(numtype,5,4)
    f[1,1] = 0.16 
    f[1,2] = 0.14587456942714338561
    f[1,3] = 0.11762370828143015682
    f[2,1] = 0.38752405202531186588
    f[2,2] = 0.15089113704380764664
    f[2,3] = -0.12805075909013044594
    f[3,1] = 1 - 2 * f[2,1] - 2 * f[1,1]
    f[3,3] = -2 * f[2,3] - 2 * f[1,3]
    flipfill!(f)
    f
end

function order6CFETOpt()
    f = zeros(numtype,5,4)
    f[1,1] = 0.1714
    f[1,2] = 0.15409059414309687213
    f[1,3] = 0.11947178242929061641
    f[2,1] = 0.37496374319946236513
    f[2,2] = 0.13813675394387646682
    f[2,3] = -0.13090674649282935743
    f[3,1] = 1 - 2 * f[2,1] - 2 * f[1,1]
    f[3,2] = 0
    f[3,3] = -2 * f[2,3] - 2 * f[1,3]
    f[1,4] = 0.07195
    f[2,4] = -0.21123356253315514306
    f[3,4] = 0
    flipfill!(f)
    f
end

function order8CFET()
    f = zeros(numtype,11,4)
    f[1,1] = df64"0.169715531043933180094151"
    f[1,2] = df64"0.152866146944615909929839"
    f[1,3] = df64"0.119167378745981369601216"
    f[1,4] = df64"0.068619226448029559107538"
    f[2,1] = df64"0.379420807516005431504230"
    f[2,2] = df64"0.148839980923180990943008"
    f[2,3] = df64"-0.115880829186628075021088" 
    f[2,4] = df64"-0.188555246668412628269760"
    f[3,1] = df64"0.469459306644050573017994"
    f[3,2] = df64"-0.379844237839363505173921"
    f[3,3] = df64"0.022898814729462898505141"
    f[3,4] = df64"0.571855043580130805495594"
    f[4,1] = df64"-0.448225927391070886302766" 
    f[4,2] = df64"0.362889857410989942809900"
    f[4,3] = df64"-0.022565582830528472333301" 
    f[4,4] = df64"-0.544507517141613383517695"
    f[5,1] = df64"-0.293924473106317605373923"
    f[5,2] = df64"-0.026255628265819381983204"
    f[5,3] = df64"0.096761509131620390100068"
    f[5,4] = df64"0.000018330145571671744069"
    f[6,1] = df64"0.447109510586798614120629"
    f[6,3] = df64"-0.200762581179816221704073"
    flipfill!(f)
    f
end

# Pre-compute Gauss quandrature and Legendre polynomial evaluations
x_g, w_g = legendre(numtype, 5)
# Transform from interval (-1, 1) to interval (0, 1) 
const x_gauss = SVector{5, numtype}((x_g .+ 1) ./ 2)
const w_gauss = SVector{5, numtype}(w_g ./ 2)
const CFET_coefs = SMatrix{11,4}(order8CFET())
const adaptive_coefs = SMatrix{5,4}(order6CFETOpt())
const order4_coefs = SMatrix{3,4}(order4CFET())
const Pls = SMatrix{5,4}(reshape([Pl(2*x_gauss[m]-1,n) for n=0:3 for m=1:5], (5,4)))

5×4 SMatrix{5, 4, Float64, 20} with indices SOneTo(5)×SOneTo(4):
 1.0  -0.90618       0.731743   -0.501031
 1.0  -0.538469     -0.0650762   0.417382
 1.0   2.22045e-16  -0.5        -3.33067e-16
 1.0   0.538469     -0.0650762  -0.417382
 1.0   0.90618       0.731743    0.501031

In [4]:
function gBfunc(t)
    #neutrongyro * B1 * cos(w * t), neutrongyro * B1 * sin(w * t), neutrongyro * B0
    neutrongyro * B1 * cos(w * t), 0.0, neutrongyro * B0
end

function dS(du, u, p, t)
    # Assigns du = dS/dt
    Bx, By, Bz = gBfunc(t)
    du[1] = u[2]*Bz - u[3]*By
    du[2] = u[3]*Bx - u[1]*Bz
    du[3] = u[1]*By - u[2]*Bx
end

function dS_static(u, p, t)
    # Uses static arrays instead
    Bx, By, Bz = gBfunc(t)
    du1 = u[2]*Bz - u[3]*By
    du2 = u[3]*Bx - u[1]*Bz
    du3 = u[1]*By - u[2]*Bx
    SA[du1,du2,du3]
end

function update_func(A,u,p,t)
    Bx, By, Bz = gBfunc(t)
    RotationVecGenerator{numtype}(-Bx, -By, -Bz)
end
u0::SV = @SVector [1.0; 0.0; 0.0]
tstart::numtype = 0.0
tend::numtype = 1e-1

0.1

In [5]:
function compute_quadrature!(t, dt, x_gauss, w_gauss, As, Pls)
    for m = 1:5
        Bx,By,Bz = gBfunc(x_gauss[m] * dt + t)
        for n = 1:4
            coef = w_gauss[m] * (2*n-1) * dt * Pls[m,n]
            As[1,n] += coef * Bx
            As[2,n] += coef * By
            As[3,n] += coef * Bz
        end
    end
end

function apply_exponentials!(u,f,As,n_exp)
    for i = n_exp:-1:1
        A = As * view(f,i,:)
        Bnorm = norm(A)
        s, c = sincos(Bnorm)
        u .= u .* c .+ (cross(u, A) .* (s/Bnorm)) .+ A .* (dot(A, u) * (1.0 - c)/(Bnorm^2))
    end
end

function CFET_fast_step!(t,dt,uprev,udummy,p,x_gauss,w_gauss,As,Pls,f,f2)
    compute_quadrature!(t, dt, x_gauss, w_gauss, As, Pls)
    apply_exponentials!(udummy,f2,As,5)
    apply_exponentials!(uprev,f,As,11)
    norm(uprev.-udummy)
end

CFET_fast_step! (generic function with 1 method)

In [15]:
function magnus_CFET_adaptive(tend::numtype, tol::numtype)
    # A pretty fast implementation of the 8-th order scheme from https://arxiv.org/pdf/1102.5071.pdf
    # The two main optimizations made here are:
    # 1. Use static arrays
    # 2. Remove unnecessary allocations (by making global variables const)
    u = MVector{3}(u0)
    u_d = MVector{3}(u0)
    u_d2 = MVector{3}(u0)
    As = @MMatrix zeros(numtype,3,4)
    p = nothing
    t = numtype(0.0)
    dt = numtype(min(tend-t,1e-4))
    f = CFET_coefs
    f2 = adaptive_coefs
    f3 = order4_coefs
    beta1 = 0.7
    beta2 = -0.4
    accept_safety = 0.81
    k = 7.0
    prev_ratio = 1.0
    count = 0
    accept_count = 0
    while t < tend
        count += 1
        As .= 0.0
        u_d .= u
        u_d2 .= u
        error = CFET_fast_step!(t, dt, u_d, u_d2, p, x_gauss, w_gauss, As, Pls, f, f2)
        ratio = tol/error

        q = ((ratio)^(beta1/k)*(prev_ratio)^(beta2/k))
        q = min(q,4.0) # control stepsize growth
        if q > accept_safety && error < 2 * tol
            accept_count += 1
            u .= u_d
            t += dt
        else
        end
        dt = min(tend - t, dt*q)
        prev_ratio = ratio
    end
    u
end

function tsit(dt)
    # Tsitouras Runge Kutta scheme
    prob = ODEProblem(dS_static, u0, (tstart, tend))
    sol = solve(prob, tsit5(), dt=dt, adaptive=false, saveat=tend)
    uend = sol.u[end]
    uend
end

function dop(tend, tol)
    # Standard DOP853 method from DifferentialEquations.jl
    # I use static arrays as well for this
    # I also enforce fixed step-size, to make performance comparisons easier
    u = SVector{3}(u0)
    prob = ODEProblem(dS_static, u, (tstart, tend))
    sol = solve(prob, DP8(), saveat=tend, reltol=0, abstol=tol)
    uend = sol.u[end]/norm(sol.u[end])
    uend
end

function vern(dt)
    # High order Verner RK scheme
    prob = ODEProblem(dS_static, u0, (tstart, tend))
    sol = solve(prob, Vern9(), dt=dt, adaptive=false, saveat=tend)
    uend = sol.u[end]
    uend
end

function exact()
    # Exact solution for rotating-wave case
    # Solution is R^T U where R takes you to the rotating frame, and U is the exponential in the rotating frame
    R = RotationVec{numtype}(0,0,-w*tend)
    U = RotationVec{numtype}(-neutrongyro*B1*tend,0,-(neutrongyro*B0+w)*tend)
    transpose(R) * U * u0
end

exact (generic function with 1 method)

In [7]:
reference_sol= vern(numtype(1e-6))

3-element SVector{3, Float64} with indices SOneTo(3):
  0.8245564642068713
 -0.5651744173430512
 -0.02616324359866901

In [8]:
norm(reference_sol - dop(tend, 1e-9))

1.4904333539048736e-8

In [9]:
norm(reference_sol - magnus_CFET_adaptive(tend, 1e-9))

5.5036636282911555e-9

In [10]:
@btime magnus_CFET_adaptive(tend, 1e-10)

  835.000 μs (4 allocations: 208 bytes)


3-element MVector{3, Float64} with indices SOneTo(3):
  0.8245564637600498
 -0.5651744179924494
 -0.026163243652036573

In [11]:
@btime dop(tend, 1e-10)

  1.221 ms (134 allocations: 14.00 KiB)


3-element SVector{3, Float64} with indices SOneTo(3):
  0.8245564642309617
 -0.5651744173645163
 -0.026163242375784056

In [ ]:
@profview magnus_CFET_adaptive(10.0, 1e-10)